层序Softmax（Hierarchical Softmax，简称 H-Softmax） 是 Word2Vec 加速训练的核心黑科技。

1. 为什么需要 H-Softmax？
痛点回顾：
在上一节的普通 Softmax 中，为了预测一个词，我们必须计算分母 $\sum{exp(u_k)}$，这意味着要和词表中所有 V 个词的向量做点积。
如果词表有 100 万个词，算一次梯度就要做 100 万次向量运算，这在工业级应用中是不可接受的。

原理：
* 我们将所有词构建成一棵二叉树（哈夫曼树）。
* 所有实词都在叶子节点。
* 所有的中间节点都是“路标”（参数向量）。
* 预测一个词的概率，就是从根节点走到该叶子节点这条路径的概率乘积。

# 第一步：准备数据与哈夫曼树辅助类

我们需要一个堆（Heap）来构建哈夫曼树，因为哈夫曼树要求高频词路径短，低频词路径长。

In [ ]:
# ============ 导入必要库 ============
import torch  # PyTorch 主库
import torch.nn as nn  # 神经网络模块
import torch.nn.functional as F  # 常用函数（如 logsigmoid）
import torch.optim as optim  # 优化器
from torch.utils.data import Dataset, DataLoader  # 数据集和数据加载器
import numpy as np  # 数值计算
import heapq  # 堆结构，用于构建哈夫曼树
import collections  # 词频统计
import requests  # 下载文本
import re  # 正则清洗文本

# ================= 数据准备 (同之前) =================
# 简化的数据获取流程
def get_alice_text():
    """下载并清洗《爱丽丝梦游仙境》文本，返回词列表"""
    url = "https://www.gutenberg.org/files/11/11-0.txt"
    try:
        text = requests.get(url).text
        # 截取正文区间，去掉版权头尾
        start = text.find("*** START OF THE PROJECT GUTENBERG EBOOK")
        end = text.find("*** END OF THE PROJECT GUTENBERG EBOOK")
        text = text[start:end].lower()
        # 只保留英文单词，去掉标点和数字
        tokens = re.findall(r'\b[a-z]+\b', text)
        return tokens
    except:
        # 若网络不可用，使用简易假数据兜底
        return ["alice", "in", "wonderland", "rabbit", "hole"] * 100

# 获取原始词序列
tokens = get_alice_text()
# 词频统计
counts = collections.Counter(tokens)
# 词到索引的映射
vocab = {w: i for i, w in enumerate(counts.keys())}
# 索引到词的映射
idx_to_word = {i: w for w, i in vocab.items()}
VOCAB_SIZE = len(vocab)
print(f"Vocab Size: {VOCAB_SIZE}")

# ================= 哈夫曼树构建 =================

class HuffmanNode:
    """哈夫曼树节点"""
    def __init__(self, left=None, right=None, idx=None, freq=0):
        self.left = left
        self.right = right
        self.idx = idx   # 叶子节点存词ID，中间节点存内部ID
        self.freq = freq # 频率，用于构建树

    # 定义比较操作符，用于优先队列（堆）
    def __lt__(self, other):
        return self.freq < other.freq

def build_huffman_tree(counts):
    """
    输入: 词频统计 {word: freq}
    输出: 树的根节点, 内部节点数量
    """
    # 1. 把所有词建成叶子节点，放入堆中
    heap = []
    for word, freq in counts.items():
        node = HuffmanNode(idx=vocab[word], freq=freq)
        heapq.heappush(heap, node)
    
    # 2. 合并节点
    # 如果有 V 个词，哈夫曼树一定有 V-1 个内部节点
    # 我们给内部节点编号，从 0 开始到 V-2
    internal_node_id = 0
    
    while len(heap) > 1:
        # 弹出频率最小的两个节点
        left = heapq.heappop(heap)
        right = heapq.heappop(heap)
        
        # 创建父节点 (内部节点)
        # 父节点频率 = 左右子节点频率之和
        parent = HuffmanNode(
            left=left,
            right=right,
            idx=internal_node_id,
            freq=left.freq + right.freq
        )
        internal_node_id += 1
        
        heapq.heappush(heap, parent)
        
    return heap[0], internal_node_id

print("Building Huffman Tree...")
root_node, num_internal_nodes = build_huffman_tree(counts)
print(f"Tree built. Internal nodes: {num_internal_nodes} (Should be V-1 approx)")

Vocab Size: 2571
Building Huffman Tree...
Tree built. Internal nodes: 2570 (Should be V-1 approx)


# 第二步：预计算路径 (Path Generation)

这一步是工业级实现的关键优化。我们不能在训练的前向传播中去遍历树，那样太慢了。我们必须提前算好每个词的路径（经过了哪些内部节点）和编码（向左还是向右），存成一个查找表。

In [ ]:
# ============ 预计算路径 (Path Generation) ============
# 存储每个词对应的路径信息
# word_idx -> (list of internal_node_ids, list of codes)
# code: 1 表示向左，-1 表示向右
word_routes = {}

def generate_paths(node, path_nodes, path_codes):
    """
    递归遍历树，记录路径
    path_nodes: 路径上经过的内部节点 ID 列表
    path_codes: 路径上的方向 (例如: [1, -1, 1...])
    """
    if node.left is None and node.right is None:
        # 叶子节点（具体的词），记录其路径
        word_routes[node.idx] = (path_nodes, path_codes)
        return

    # 向左走 (Code = 1)
    # 左子树为正类(1)，右子树为负类(-1)，方便计算 log_sigmoid(score * code)
    if node.left:
        generate_paths(node.left, path_nodes + [node.idx], path_codes + [1])
    
    # 向右走 (Code = -1)
    if node.right:
        generate_paths(node.right, path_nodes + [node.idx], path_codes + [-1])

# 从根节点开始生成所有词的路径
generate_paths(root_node, [], [])
print("Paths generated for all words.")

# 检查 "alice" 的路径
alice_id = vocab['alice']
alice_path, alice_codes = word_routes[alice_id]
print(f"Path for 'alice': Nodes={alice_path}, Codes={alice_codes}")

Paths generated for all words.
Path for 'alice': Nodes=[2569, 2567, 2564, 2558, 2546, 2524], Codes=[1, -1, -1, 1, -1, -1]


# 第三步：Dataset 修改

因为每个词的路径长度不一样，PyTorch 的 DataLoader 默认要求通过 collate_fn 把样本堆叠成 Tensor，这要求维度一致。

为了处理变长路径，我们有两个选择：

1. Padding（填充）：把所有路径填补到最大深度（最简单）。
2. 自定义 Collate：稍微复杂点。

在这里我们使用padding.

In [ ]:
# ============ Dataset 修改（支持变长路径） ============
# 计算树的最大深度，用于 padding
MAX_DEPTH = len(max(word_routes.values(), key=lambda x: len(x[0]))[0])
print(f"Max tree depth: {MAX_DEPTH}")

class CBOWHSDataset(Dataset):
    """CBOW + Hierarchical Softmax 的数据集"""
    def __init__(self, token_ids, window_size):
        self.data = []
        for i in range(window_size, len(token_ids) - window_size):
            # 目标词
            target_id = token_ids[i]
            # 上下文：左 window_size + 右 window_size
            context = token_ids[i-window_size:i] + token_ids[i+1:i+window_size+1]
            
            # 获取路径信息
            path_nodes, path_codes = word_routes[target_id]
            path_len = len(path_nodes)
            
            # Padding: 用 0 填充到最大深度
            # 注意：用 mask 来忽略填充部分
            padded_nodes = path_nodes + [0] * (MAX_DEPTH - path_len)
            padded_codes = path_codes + [0] * (MAX_DEPTH - path_len)
            
            self.data.append((
                torch.tensor(context),
                torch.tensor(padded_nodes),
                torch.tensor(padded_codes).float(),
                torch.tensor(path_len)  # 记录真实长度，用于 mask
            ))
            
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

# 转换 token 列表为索引列表
token_ids = [vocab[t] for t in tokens]
# 创建数据集
dataset = CBOWHSDataset(token_ids, window_size=2)
# 创建 DataLoader，自动批处理并打乱
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

Max tree depth: 15


# 第四步：构建 H-Softmax 模型

* 输入：Context Words
* 中间节点参数：我们把树的内部节点也看作一种 Embedding！
* 损失计算：我们不再计算 Softmax。
  * 我们在路径上的每一个节点做一个二分类（Binary Classification）。
  * 目标是最大化路径概率：$P(w) = \prod P(\text{node}_i)$
  * 取对数后：$\log P(w) = \sum \log \sigma(h^T \cdot \theta_i \cdot \text{code}_i)$
  * 其中 $\theta_i$ 是当前内部节点的向量，code 取 1 或 -1。

In [ ]:
class CBOW_HierarchicalSoftmax(nn.Module):
    """CBOW + Hierarchical Softmax 模型"""
    def __init__(self, vocab_size, num_internal_nodes, embed_dim):
        super().__init__()
        # 1. 输入词向量 (用于上下文)
        self.in_embeddings = nn.Embedding(vocab_size, embed_dim)
        
        # 2. 内部节点向量 (树上的路标)
        # 注意：这里不是 vocab_size，而是内部节点数量
        self.internal_node_embeddings = nn.Embedding(num_internal_nodes, embed_dim)
        
    def forward(self, context_ids, path_nodes, path_codes, path_lens):
        """
        context_ids: [batch, 2*window]
        path_nodes:  [batch, max_depth]  (目标词路径上的节点ID)
        path_codes:  [batch, max_depth]  (方向: 1 或 -1)
        path_lens:   [batch]             (真实路径长度)
        """
        # A. 计算上下文向量 h
        # [batch, 2*window, dim] -> [batch, dim]
        context_embeds = self.in_embeddings(context_ids)
        h = torch.mean(context_embeds, dim=1)
        
        # B. 获取路径上所有节点的向量 theta
        # [batch, max_depth, dim]
        path_node_vecs = self.internal_node_embeddings(path_nodes)
        
        # C. 计算 h 和 theta 的点积
        # 扩展 h 的维度以便广播: [batch, 1, dim]
        h_expanded = h.unsqueeze(1)
        # 点积: [batch, max_depth]
        raw_scores = torch.sum(h_expanded * path_node_vecs, dim=2)
        
        # D. 引入方向码 (path_codes)
        # 左(1)：log_sigmoid(score)
        # 右(-1)：log_sigmoid(-score)
        directed_scores = raw_scores * path_codes
        
        # E. 计算 Log Sigmoid
        log_probs = F.logsigmoid(directed_scores)
        
        # F. Mask 掉填充部分
        batch_size, max_len = path_nodes.size()
        # mask: [batch, max_depth]
        mask = torch.arange(max_len).expand(batch_size, max_len).to(path_nodes.device) < path_lens.unsqueeze(1)
        valid_log_probs = log_probs * mask.float()
        
        # Sum over path, Mean over batch
        # 负号表示最小化负对数似然
        loss = -torch.sum(valid_log_probs) / batch_size
        return loss

# 实例化模型
EMBED_DIM = 50
model = CBOW_HierarchicalSoftmax(VOCAB_SIZE, num_internal_nodes, EMBED_DIM)
print(model)

CBOW_HierarchicalSoftmax(
  (in_embeddings): Embedding(2571, 50)
  (internal_node_embeddings): Embedding(2570, 50)
)


# 第五步：训练循环

In [ ]:
# ============ 训练循环 ============
optimizer = optim.Adam(model.parameters(), lr=0.003)
epochs = 10

# 如果有 GPU 则用 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Starting H-Softmax Training...")
for epoch in range(epochs):
    total_loss = 0.0
    for batch in dataloader:
        # 解包数据，并放到同一设备上
        context_ids, nodes, codes, lens = [x.to(device) for x in batch]
        
        # 清空梯度
        optimizer.zero_grad()
        
        # 前向传播直接返回 Loss
        loss = model(context_ids, nodes, codes, lens)
        
        # 反向传播
        loss.backward()
        
        # 更新参数
        optimizer.step()
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch}: Avg Loss = {avg_loss:.4f}")

print("Training Done!")

Starting H-Softmax Training...
Epoch 0: Avg Loss = 10.9936
Epoch 1: Avg Loss = 7.9999
Epoch 2: Avg Loss = 6.6912
Epoch 3: Avg Loss = 5.9471
Epoch 4: Avg Loss = 5.4612
Epoch 5: Avg Loss = 5.1137
Epoch 6: Avg Loss = 4.8465
Epoch 7: Avg Loss = 4.6297
Epoch 8: Avg Loss = 4.4481
Epoch 9: Avg Loss = 4.2905
Training Done!
